# 🌍 FineTome → Kiswahili | Qwen3.5-4B Local Translation
### Kaggle 2×T4 · Free · No API Cost · HuggingFace Hub Push

**Model:** `Qwen/Qwen3.5-4B-Instruct` (local, free)  
**Source:** `mlabonne/FineTome-100k`  
**Output:** `lengai-lab/FineTome-20k-sw`  
**Cost:** $0.00  
**Time:** ~8–12 hours on Kaggle T4 (run overnight)

---
```
1. Install & load model (Qwen3.5-4B-Instruct, 4-bit)
2. Load & filter FineTome → best 20K rows
3. Translate with batched vLLM inference
4. Quality validation
5. Save checkpoints (crash-safe)
6. Push to HuggingFace Hub
```

> **Kaggle tip:** Enable both T4 GPUs — Settings → Accelerator → GPU T4 x2

## 1. Install Dependencies

In [ ]:
%%capture
!pip install transformers accelerate bitsandbytes datasets huggingface_hub tqdm

## 2. Configuration

In [ ]:
import os

# ── KEYS ─────────────────────────────────────────────────────────────────────
HF_TOKEN       = os.environ.get("HF_TOKEN")  # Kaggle → Add-ons → Secrets

# ── CONFIG ───────────────────────────────────────────────────────────────────
MODEL_NAME     = "Qwen/Qwen3.5-4B-Instruct"
SOURCE_DATASET = "mlabonne/FineTome-100k"
TARGET_ROWS    = 20_000
HF_REPO        = "lengai-lab/FineTome-20k-sw"   # ← your HF username

# Translation batch size — tune based on VRAM
# T4 16GB: batch=4 safe | 2×T4 32GB: batch=8
BATCH_SIZE     = 4
MAX_NEW_TOKENS = 1024
TEMPERATURE    = 0.3   # Low = faithful translation

# Checkpoint — saves every N rows (crash safety)
CHECKPOINT_DIR  = "/kaggle/working/checkpoints"
CHECKPOINT_EVERY = 500

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✅ Config ready")
print(f"   Model      : {MODEL_NAME}")
print(f"   Target rows: {TARGET_ROWS:,}")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Checkpoint : every {CHECKPOINT_EVERY} rows")

## 3. Load Qwen3.5-4B-Instruct (4-bit)

4-bit quantization keeps VRAM under 6GB — leaves plenty of room for batched inference.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Check GPUs
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory/1e9:.1f} GB")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

print(f"\nLoading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",   # Spreads across both T4s automatically
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)
model.eval()

print(f"\n✅ Model loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 4. Load & Filter FineTome

In [ ]:
from datasets import load_dataset
import re, random, json

print("Loading FineTome-100k...")
ds = load_dataset(SOURCE_DATASET, split="train")
print(f"Loaded {len(ds):,} rows")

def extract_turns(row):
    """Extract first human/gpt turn from ShareGPT format."""
    instruction, output = "", ""
    for turn in row.get("conversations", []):
        if turn["from"] in ("human", "user") and not instruction:
            instruction = turn["value"].strip()
        elif turn["from"] in ("gpt", "assistant") and not output:
            output = turn["value"].strip()
    return instruction, output

def is_code_heavy(text, threshold=0.3):
    patterns = [
        r"```[\s\S]*?```",
        r"def \w+\s*\(",
        r"import \w+",
        r"class \w+[:(]",
        r"<[a-zA-Z][^>]*>[^<]*</[a-zA-Z]+>",
    ]
    code_chars = sum(len(m.group()) for p in patterns for m in re.finditer(p, text))
    return (code_chars / max(len(text), 1)) > threshold

def is_translatable(instruction, output):
    if not instruction or not output:       return False
    if len(output.split()) < 20:            return False
    if len(output.split()) > 500:           return False  # Keep shorter for faster inference
    if is_code_heavy(instruction + output): return False
    return True

print("Filtering...")
filtered = []
for i, row in enumerate(ds):
    instruction, output = extract_turns(row)
    if is_translatable(instruction, output):
        filtered.append({
            "id":          str(i),
            "instruction": instruction,
            "output":      output,
        })

# Evenly spaced sample for topic diversity
random.seed(42)
if len(filtered) > TARGET_ROWS:
    step     = len(filtered) // TARGET_ROWS
    selected = [filtered[i] for i in range(0, len(filtered), step)][:TARGET_ROWS]
else:
    selected = filtered

print(f"After filtering : {len(filtered):,}")
print(f"Selected        : {len(selected):,} rows for translation")

## 5. Translation Engine

Batched inference with checkpoint saving every 500 rows — safe to interrupt and resume.

In [ ]:
# ── TRANSLATION PROMPT ───────────────────────────────────────────────────────
SYSTEM_PROMPT = """Wewe ni mtafsiri mtaalamu wa Kiingereza hadi Kiswahili sanifu.
Kazi yako ni kutafsiri maagizo na majibu kwa Kiswahili fasaha na asilia.

Kanuni:
1. Tumia Kiswahili sanifu — si tafsiri ya neno kwa neno
2. Hifadhi maana kamili bila kupoteza taarifa yoyote
3. Maneno ya kitaalamu (AI, model, data, algorithm) yanaweza kubaki Kiingereza
4. Nambari na fomula za hisabati vibaki kama vilivyo
5. Jibu kwa JSON tu — bila maelezo, bila utangulizi

Muundo wa jibu (JSON tu):
{"instruction": "<tafsiri>", "output": "<tafsiri>"}"""

def make_prompt(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content":
            f"Tafsiri yafuatayo kwa Kiswahili:\n\n"
            f"MAAGIZO:\n{row['instruction']}\n\n"
            f"JIBU:\n{row['output']}"
        },
    ]
    # Apply Qwen3.5 chat template
    # enable_thinking=False → faster, no <think> block in translation
    return tokenizer.apply_chat_template(
        messages,
        tokenize            = False,
        add_generation_prompt = True,
        enable_thinking     = False,
    )

def parse_translation(text):
    """Extract JSON from model output — handles markdown fences and extra text."""
    # Strip <think>...</think> block if present
    text = re.sub(r"<think>[\s\S]*?</think>", "", text).strip()
    # Try direct JSON parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Strip markdown fences
    text = re.sub(r"```(?:json)?\s*", "", text).strip().rstrip("`")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Try to find JSON object in text
    match = re.search(r"\{[\s\S]*\}", text)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None

print("✅ Translation engine ready")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Max tokens : {MAX_NEW_TOKENS}")

# Quick test on 1 sample
test_prompt = make_prompt(selected[0])
print(f"\nTest prompt (first 300 chars):\n{test_prompt[:300]}...")

## 6. Run Translation (Crash-Safe)

Saves a checkpoint every 500 rows. If Kaggle disconnects, re-run this cell — it picks up from the last checkpoint automatically.

In [ ]:
import time
from tqdm import tqdm

CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, "translated.jsonl")
PROGRESS_FILE   = os.path.join(CHECKPOINT_DIR, "progress.txt")

# ── RESUME FROM CHECKPOINT ───────────────────────────────────────────────────
translated_rows = []
start_idx       = 0

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                translated_rows.append(json.loads(line))
    start_idx = len(translated_rows)
    print(f"♻️  Resuming from checkpoint: {start_idx:,} rows already done")
else:
    print("🆕 Starting fresh translation run")

# Rows remaining
remaining = selected[start_idx:]
print(f"   Remaining : {len(remaining):,} rows")
print(f"   ETA       : ~{len(remaining) * 3 / 3600:.1f} hours (est. 3s/row on T4)")

# ── BATCH TRANSLATE ──────────────────────────────────────────────────────────
errors    = []
t_start   = time.time()

for batch_start in tqdm(range(0, len(remaining), BATCH_SIZE), desc="Translating"):
    batch = remaining[batch_start : batch_start + BATCH_SIZE]

    # Build prompts
    prompts = [make_prompt(row) for row in batch]

    # Tokenize with padding
    inputs = tokenizer(
        prompts,
        return_tensors = "pt",
        padding        = True,
        truncation     = True,
        max_length     = 2048,
    ).to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            temperature    = TEMPERATURE,
            do_sample      = True,
            top_p          = 0.9,
            pad_token_id   = tokenizer.eos_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )

    # Decode — only new tokens (strip input prompt)
    input_len = inputs["input_ids"].shape[1]
    for i, (row, output_ids) in enumerate(zip(batch, outputs)):
        new_tokens = output_ids[input_len:]
        raw_text   = tokenizer.decode(new_tokens, skip_special_tokens=True)
        parsed     = parse_translation(raw_text)

        if parsed and parsed.get("instruction") and parsed.get("output"):
            translated_rows.append({
                "instruction":    parsed["instruction"].strip(),
                "output":         parsed["output"].strip(),
                "instruction_en": row["instruction"],
                "output_en":      row["output"],
                "source":         "FineTome-100k",
                "lang":           "sw",
            })
        else:
            errors.append({"id": row["id"], "raw": raw_text[:200]})


    # ── CHECKPOINT SAVE ──────────────────────────────────────────────────────
    total_done = start_idx + batch_start + len(batch)
    if total_done % CHECKPOINT_EVERY < BATCH_SIZE:
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            for r in translated_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        elapsed = (time.time() - t_start) / 60
        speed   = len(translated_rows) / max(time.time() - t_start, 1)
        eta_min = (len(remaining) - batch_start) / max(speed * 60, 1)
        tqdm.write(f"  💾 Checkpoint saved: {len(translated_rows):,} rows | " f"{elapsed:.0f} min elapsed | ETA {eta_min:.0f} min")


# Final save
with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
    for r in translated_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

elapsed = (time.time() - t_start) / 60
print(f"\n Translation complete")
print(f"   Translated : {len(translated_rows):,}")
print(f"   Errors     : {len(errors):,}")
print(f"   Time       : {elapsed:.1f} min")
print(f"   Speed      : {len(translated_rows)/elapsed:.0f} rows/min")

## 7. Quality Validation

In [ ]:
import re

# Load from checkpoint if needed
if not translated_rows:
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        translated_rows = [json.loads(l) for l in f if l.strip()]
    print(f"Loaded {len(translated_rows):,} rows from checkpoint")

SW_MARKERS = [
    r'\bni\b', r'\bna\b', r'\bya\b', r'\bwa\b', r'\bkwa\b',
    r'\bkuwa\b', r'\bkama\b', r'\bkatika\b', r'\blakini\b',
    r'\bpia\b', r'\bsana\b', r'\bwakati\b', r'\bkwa hivyo\b',
    r'\bkwa mfano\b', r'\bkwa sababu\b',
]

def has_swahili(text):
    t = text.lower()
    return sum(1 for p in SW_MARKERS if re.search(p, t)) >= 2

def length_ratio(src, tgt):
    return len(tgt.split()) / max(len(src.split()), 1)

def validate(row):
    instr_sw = row["instruction"]
    out_sw   = row["output"]
    out_en   = row["output_en"]

    if not instr_sw or not out_sw:
        return False, "empty"
    if not has_swahili(instr_sw + " " + out_sw):
        return False, "no_swahili"
    ratio = length_ratio(out_en, out_sw)
    if not (0.5 <= ratio <= 2.5):
        return False, f"bad_ratio_{ratio:.2f}"
    if out_sw.strip() == out_en.strip():
        return False, "untranslated"
    return True, "ok"

passed, failed = [], []
fail_reasons   = {}

for row in translated_rows:
    ok, reason = validate(row)
    if ok:
        passed.append(row)
    else:
        failed.append(row)
        fail_reasons[reason] = fail_reasons.get(reason, 0) + 1

print(f"✅ Quality gate")
print(f"   Passed : {len(passed):,} ({100*len(passed)/max(len(translated_rows),1):.1f}%)")
print(f"   Failed : {len(failed):,}")
if fail_reasons:
    print(f"   Fail reasons: {fail_reasons}")

# Show 3 samples
print("\n" + "=" * 60)
for s in passed[:3]:
    print(f"\nEN instruction : {s['instruction_en'][:100]}")
    print(f"SW instruction : {s['instruction'][:100]}")
    print(f"EN output      : {s['output_en'][:150]}")
    print(f"SW output      : {s['output'][:150]}")
    print("-" * 60)

## 8. Push to HuggingFace Hub

In [ ]:
from datasets import Dataset
from huggingface_hub import login

login(token=HF_TOKEN)

# Version 1 — Alpaca-style (instruction / output)
ds_main = Dataset.from_list(passed)

# Version 2 — ShareGPT (direct Unsloth SFT use)
ds_sharegpt = Dataset.from_list([
    {
        "conversations": [
            {"from": "human", "value": r["instruction"]},
            {"from": "gpt",   "value": r["output"]},
        ],
        "lang":   "sw",
        "source": r["source"],
    }
    for r in passed
])

print(f"Pushing {len(passed):,} rows...")

ds_main.push_to_hub(
    HF_REPO,
    token          = HF_TOKEN,
    commit_message = f"Add {len(passed):,} Swahili translations via Qwen3.5-4B",
)
ds_sharegpt.push_to_hub(
    HF_REPO + "-sharegpt",
    token          = HF_TOKEN,
    commit_message = "ShareGPT format — ready for Unsloth SFT",
)

print(f"\n🎉 Dataset published!")
print(f"   Main     → https://huggingface.co/datasets/{HF_REPO}")
print(f"   ShareGPT → https://huggingface.co/datasets/{HF_REPO}-sharegpt")
print(f"\n⚡ Next: finetune Gemma4 E4B on this dataset")
print(f"   load_dataset('{HF_REPO}-sharegpt')")

## Notes

### Crash recovery
If Kaggle disconnects mid-run, just re-run Cell 6 — it reads from `checkpoints/translated.jsonl` and resumes from where it left off. No rows are lost.

### Speed tuning
```python
BATCH_SIZE = 4   # T4 x1 — safe default
BATCH_SIZE = 8   # T4 x2 — faster, watch VRAM
BATCH_SIZE = 2   # If you hit OOM
```

### Output schema
```
instruction    : str  — Swahili instruction
output         : str  — Swahili response
instruction_en : str  — Original English instruction
output_en      : str  — Original English response
source         : str  — "FineTome-100k"
lang           : str  — "sw"
```

### After this:
→ `gemma4_e4b_swahili_sft.ipynb` — finetune Gemma4 E4B on this dataset  
→ `qwen35_4b_swahili_sft.ipynb`  — finetune Qwen3.5-4B on this dataset

---
**Lengai AI Lab — Swahili LLM Research Track** 🌍